# TCGA-LGG Lightweight Extensions (Phase 3)

Trains lightweight U-Net variants, distills, quantizes, and runs the
equity-of-compression analysis. Requires a **GPU** accelerator (T4).

**Before running:** attach the TCGA-LGG dataset and turn Accelerator = GPU T4,
Internet = On. Then Run All (or run cells top to bottom).

In [ ]:
# Cell 1 - clone repo and locate the data
import os
!git clone -q https://github.com/adithyabalakumar007-ai/mlrc-tracks-TCGA-LGG-comm.git
EXT = '/kaggle/working/mlrc-tracks-TCGA-LGG-comm/extensions'
print('extensions present:', os.path.isdir(EXT))

DATA_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    if os.path.basename(root) == 'kaggle_3m':
        DATA_PATH = root
        break
print('DATA_PATH =', DATA_PATH)
assert DATA_PATH, 'kaggle_3m not found - attach the dataset in the right panel'

In [ ]:
# Cell 2 - config
EPOCHS = 25   # lower (e.g. 15) if short on GPU time; raise toward 30 for best Dice
import torch
print('CUDA available:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# Cell 3 - train Full U-Net (teacher).  ~40-60 min on T4
!python {EXT}/lightweight_unet.py --mode train --variant full  --datapath "{DATA_PATH}" --epochs {EPOCHS}

In [ ]:
# Cell 4 - train Slim U-Net
!python {EXT}/lightweight_unet.py --mode train --variant slim  --datapath "{DATA_PATH}" --epochs {EPOCHS}

In [ ]:
# Cell 5 - train Micro U-Net
!python {EXT}/lightweight_unet.py --mode train --variant micro --datapath "{DATA_PATH}" --epochs {EPOCHS}

In [ ]:
# Cell 6 - knowledge distillation: Full (teacher) -> Micro (student)
!python {EXT}/distill.py --teacher_path models/unet_full_best.pth --student micro --datapath "{DATA_PATH}" --epochs {EPOCHS}

In [ ]:
# Cell 7 - post-training quantization (FP16 + INT8) of Slim and Micro
!python {EXT}/quantize_model.py --model_path models/unet_slim_best.pth  --variant slim  --datapath "{DATA_PATH}"
!python {EXT}/quantize_model.py --model_path models/unet_micro_best.pth --variant micro --datapath "{DATA_PATH}"

In [ ]:
# Cell 8 - architecture benchmark (params / size / GPU+CPU latency)
!python {EXT}/lightweight_unet.py --mode benchmark

In [ ]:
# Cell 9 - equity of compression: Dice vs tumor size for every trained model
!python {EXT}/compression_equity.py --datapath "{DATA_PATH}"

In [ ]:
# Cell 10 - bundle all results + figures into one downloadable zip
import shutil, glob, os
os.makedirs('/kaggle/working/ext_bundle/results', exist_ok=True)
os.makedirs('/kaggle/working/ext_bundle/figures', exist_ok=True)
for f in glob.glob('/kaggle/working/results/*'):
    shutil.copy(f, '/kaggle/working/ext_bundle/results/')
for f in glob.glob('/kaggle/working/figures/*'):
    shutil.copy(f, '/kaggle/working/ext_bundle/figures/')
shutil.make_archive('/kaggle/working/ext_bundle', 'zip', '/kaggle/working/ext_bundle')
print('Bundle ready -> /kaggle/working/ext_bundle.zip')
print('Files:')
for f in sorted(glob.glob('/kaggle/working/ext_bundle/**/*', recursive=True)):
    print(' ', f)